# El Espacio qang en Criptografía Cuántica: Diagnóstico y Auditoría de Falsación

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_criptografia_colab.ipynb)

Este cuaderno implementa las aplicaciones del espacio $qg$ a la **Criptografía Cuántica (QKD BB84 y QRNG)** según los capítulos correspondientes de la Parte VII de la obra *Computación, información y física cuántica en el espacio qang* (35 capítulos).

### Delimitación Científica Estricta
* **Criptografía Post-Cuántica (PQC - Dilithium, Kyber, Retículos):** Algoritmos clásicos basados en dureza matemática. El marco $qg$ **no** aplica ni añade ventajas de ruptura.
* **Criptografía Cuántica (QKD / QRNG):** Los observables físicos son magnitudes angulares-probabilísticas ($qg_Z, qg_X$). Aquí el espacio $qg$ proporciona una herramienta de **diagnóstico y discriminación de ruido vs. espionaje**, no un nuevo teorema de secreto incondicional.

In [ ]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang>=0.14.0", "numpy", "scipy", "matplotlib"], check=False)
importlib.invalidate_caches()

import numpy as np
import qang
from qang import formulation as F

print(f"Módulo de diagnóstico criptográfico cargado: qang v{qang.__version__}")

In [ ]:
# 1. BB84: Discriminación entre Ruido Térmico T1 y Ataque de Interceptación (Cap. 26-27, §40)
# A_Z = qg(0) + qg(1) = 2[e(1->0) - e(0->1)]
# El deterioro natural de T1 produce asimetría A_Z > 0.
# La interceptación de Eve produce errores simétricos: A_Z permanece invariante.

def sim_bb84_channel(gamma_t1=0.02, p_dephasing=0.01, eve_fraction=0.0):
    e01 = 0.005 + 0.25 * eve_fraction
    e10 = 0.005 + gamma_t1 + 0.25 * eve_fraction
    qz = 0.5 * (e01 + e10)
    az = 2.0 * (e10 - e01)
    return e01, e10, qz, az

print(f"{'Escenario':<30}{'e(0->1)':>10}{'e(1->0)':>10}{'QBER':>10}{'Testigo A_Z':>14}")
scenarios = [
    ("Línea Base (Hardware Sano)", 0.02, 0.01, 0.0),
    ("Deriva T1 (gamma=0.05)", 0.05, 0.01, 0.0),
    ("Espía Eve (f=0.05)", 0.02, 0.01, 0.05),
    ("Espía Eve (f=0.15)", 0.02, 0.01, 0.15),
]
for name, g, p, f in scenarios:
    e01, e10, qz, az = sim_bb84_channel(g, p, f)
    print(f"{name:<30}{e01:10.4f}{e10:10.4f}{qz:10.4f}{az:+14.4f}")

In [ ]:
# 2. Generación Cuántica de Números Aleatorios (QRNG) Certificada (Cap. 27, §41)
# H_min(Z|E) = -log2((1 + sqrt(1 - r_perp^2)) / 2) con r_perp = sqrt(qg_X^2 + qg_Y^2)
def min_entropy_certified(qg_x, qg_y):
    r_perp = np.sqrt(qg_x**2 + qg_y**2)
    r_perp = min(r_perp, 1.0)
    return -np.log2((1.0 + np.sqrt(max(0.0, 1.0 - r_perp**2))) / 2.0)

print("Entropía mínima privada certificada frente a decoherencia:")
for r in (1.0, 0.8, 0.5, 0.2, 0.0):
    h_min = min_entropy_certified(r, 0.0)
    print(f"Coherencia ecuatorial r_perp = {r:.2f} -> H_min(Z|E) = {h_min:.4f} bits/disparo")